In [ ]:
#!/usr/bin/env python3
"""
Simple Poisson augmentation script.
Edit INPUT_DIR and OUTPUT_DIR and run:
    python poisson.py

Dependencies:
    pip install opencv-python scikit-image numpy
"""
import os
import cv2
import numpy as np
from skimage.util import random_noise
from skimage import img_as_ubyte

from google.colab import files
files.upload()
!unzip dataset_images.zip
# ---------- user settings (change these) ----------
INPUT_DIR  = "/content/dataset_images"
OUTPUT_DIR = "/content/poisson-output-new"
import shutil
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Recommended "good-quality" scales from your results
SCALES = [0.49, 0.68, 0.87, 1.06, 1.24, 1.43]  # 6 good-quality variants
# optional add 0.30 if you want one more moderate variant:
# SCALES = [0.30, 0.49, 0.68, 0.87, 1.06, 1.24, 1.43]

ALLOWED_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
# --------------------------------------------------

os.makedirs(OUTPUT_DIR, exist_ok=True)

def read_and_normalize(path):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise RuntimeError(f"Cannot read: {path}")

    # ensure 3 channels
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    elif img.ndim == 3 and img.shape[2] == 1:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

    # normalize per-image to 0..1 float32
    img = img.astype(np.float32)
    mn, mx = img.min(), img.max()
    if mx - mn > 0:
        img = (img - mn) / (mx - mn)
    else:
        img = np.zeros_like(img, dtype=np.float32)

    # convert BGR->RGB for skimage
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

def save_rgb_float_as_png(path, rgb_float):
    uint8 = img_as_ubyte(rgb_float)             # scales to 0..255
    bgr = cv2.cvtColor(uint8, cv2.COLOR_RGB2BGR)
    cv2.imwrite(path, bgr)

files = sorted([f for f in os.listdir(INPUT_DIR) if os.path.splitext(f)[1].lower() in ALLOWED_EXTS])

if not files:
    print("No images found in", INPUT_DIR)
else:
    print(f"Found {len(files)} images. Generating {len(SCALES)} Poisson variants each -> {len(files)*len(SCALES)} images.")

for fname in files:
    in_path = os.path.join(INPUT_DIR, fname)
    try:
        img = read_and_normalize(in_path)
    except Exception as e:
        print("Skipping", fname, ":", e)
        continue

    base = os.path.splitext(fname)[0]
    for i, scale in enumerate(SCALES, start=1):
        # scale to simulate photon counts, apply Poisson, then rescale back
        scaled = img * scale
        noisy = random_noise(scaled, mode="poisson")
        noisy = np.clip(noisy / scale, 0.0, 1.0)
        out_name = f"{base}_poisson_{i:02d}_scale_{scale:.2f}.png"
        out_path = os.path.join(OUTPUT_DIR, out_name)
        save_rgb_float_as_png(out_path, noisy)

print("Done.")


Saving dataset_images.zip to dataset_images (1).zip
Archive:  dataset_images.zip
   creating: dataset_images/
  inflating: dataset_images/slide5_350_10-14_R1.jpg  
  inflating: dataset_images/slide5_350_11-14_R1.jpg  
  inflating: dataset_images/slide5_350_12-14_R1.jpg  
  inflating: dataset_images/slide5_350_8-14_R1.jpg  
  inflating: dataset_images/slide5_350_9-14_R1.jpg  
  inflating: dataset_images/Chip7_IgG_HC_10-14_R1.jpg  
  inflating: dataset_images/Chip7_IgG_HC_11-14_R1.jpg  
  inflating: dataset_images/Chip7_IgG_HC_12-14_R1.jpg  
  inflating: dataset_images/Chip7_IgG_HC_13-14_R1.jpg  
  inflating: dataset_images/Chip7_IgG_HC_14-14_R1.jpg  
  inflating: dataset_images/Chip7_IgG_HC_9-14_R1.jpg  
  inflating: dataset_images/Chip8_IgG_FC_10-14_R1.jpg  
  inflating: dataset_images/Chip8_IgG_FC_1-14_R1.jpg  
  inflating: dataset_images/Chip8_IgG_FC_2-14_R1.jpg  
  inflating: dataset_images/Chip8_IgG_FC_3-14_R1.jpg  
  inflating: dataset_images/Chip8_IgG_FC_4-14_R1.jpg  
  inflating

In [ ]:
!zip -r poisson-output-new.zip poisson-output-new
from google.colab import files
files.download("poisson-output-new.zip")

  adding: poisson-output-new/ (stored 0%)
  adding: poisson-output-new/Chip8_IgG_FC_6-14_R1_poisson_05_scale_1.24.png (deflated 1%)
  adding: poisson-output-new/slide5_350_11-14_R1_poisson_02_scale_0.68.png (deflated 0%)
  adding: poisson-output-new/Chip8_IgG_FC_9-14_R1_poisson_06_scale_1.43.png (deflated 1%)
  adding: poisson-output-new/Chip7_IgG_HC_10-14_R1_poisson_03_scale_0.87.png (deflated 1%)
  adding: poisson-output-new/Chip7_IgG_HC_14-14_R1_poisson_02_scale_0.68.png (deflated 1%)
  adding: poisson-output-new/Chip8_IgG_FC_1-14_R1_poisson_06_scale_1.43.png (deflated 1%)
  adding: poisson-output-new/Chip7_IgG_HC_13-14_R1_poisson_02_scale_0.68.png (deflated 1%)
  adding: poisson-output-new/Chip7_IgG_HC_9-14_R1_poisson_06_scale_1.43.png (deflated 1%)
  adding: poisson-output-new/Chip8_IgG_FC_5-14_R1_poisson_04_scale_1.06.png (deflated 1%)
  adding: poisson-output-new/slide5_350_10-14_R1_poisson_02_scale_0.68.png (deflated 1%)
  adding: poisson-output-new/Chip8_IgG_FC_1-14_R1_poisson

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>